# HeatUp 01 - prepare

Puts every structure on disk and writes down **what is to be computed** for it (`plan.json` next to each `POSCAR`). No GPU needed.

| notebook | does | needs |
|---|---|---|
| **01 prepare** (this one) | finds competing phases, writes the plan and the input cells of every calculation | nothing (a calculator only to relax) |
| **02 compute** (optional) | runs the plans with an interatomic potential | GPU/CPU + potential |
| **03 analysis** | reads whatever is on disk and judges it | nothing |

Skip 02 if you compute with another code (DFT) in the same folders. This notebook sets only what changes between projects; every other
setting is in `heatup/core/config.py` (`cfg.show("md")` prints a section).

**You ask for checks; everything they need is planned:**

| check (`cfg.plan.gates`) | calculations planned |
|---|---|
| `mechanical` | `elastic` |
| `structural_transition` | `md` |
| `vibrational` | the steps of `cfg.vibrational.sources`: `harmonic`, `quasiharmonic`, `md` |
| `thermodynamic` | the steps of `cfg.terms.vibrational`, for the target **and** its competing phases |

`relax` is always needed. Leave `"relax"` out of the gates if your structures are already relaxed (other code or model): they are then used as given.

```
database/<material>/<label>/        POSCAR  plan.json  metadata.json
    relaxation/  elastic/  md/<T>K/  stability/
    phonons/{harmonic, quasiharmonic, perturbative, anharmonic}/
```
Structures live in `candidates/` and are copied to `database/` when needed. Competing phases already in the database are reused only if they were
computed with the settings of this project.

In [ ]:
import json
import warnings
import pandas as pd
from IPython.display import display

import heatup
from heatup import cfg, export_inputs, prepare_project, preview_plan, run_plan, save_project
from heatup.analyze.report import print_layout, show_status

warnings.filterwarnings("ignore")
heatup.verbose()
print("HeatUp", heatup.__version__)

## Configuration
The only cell you edit.

In [ ]:
# -- Project ------------------------------------------------------------------------------------------
cfg.calc.model = "mace-mpa-0-medium.model"       # the potential (file or mace_mp name); notebook 02 uses the same one
cfg.phases.sources = ["database", "candidates"]  # where competing phases come from; add "mp-api" (needs $MP_API_KEY) to download them

# -- What to check ------------------------------------------------------------------------------------
cfg.plan.gates = ["relax", "mechanical", "structural_transition", "vibrational", "thermodynamic"]  # no "relax": structures used as given
cfg.vibrational.sources = ["harmonic", "anharmonic"]   # evidence of the vibrational check: "harmonic" | "quasiharmonic" | "anharmonic" (MD)

# -- Free energy of the hull: G(T) = H0 + the terms that are not False, the same for every phase -------
cfg.terms.vibrational = "anharmonic"   # False (static 0 K hull) | "ZPE" | "harmonic" | "quasiharmonic" | "anharmonic" (from MD, experimental)
# electronic, magnetic, configurational, orientational and defects are False unless set here; contributions.show() lists their models

# -- Cost of the MD (the other calculations use the defaults of config.py) -------------------------------
cfg.md.supercell = (2, 2, 2)   # explicit supercell (None: automatic, from cfg.md.min_cell_A)
cfg.md.min_cell_A = None       # must be None while a supercell is given
cfg.md.n_steps = 300           # x 1 fs: a quick test; use 30_000 (30 ps) for production

# -- Targets ------------------------------------------------------------------------------------------
# "symmetry" = label of the structure folder; "temperature" = operating T (and MD T), a number or a list
RUN_LIST = [{"material": "BaTiO3", "symmetry": mp, "temperature": 300} for mp in [
    "mp-1076932", "mp-5020", "mp-5777", "mp-644497", "mp-995191", "mp-5933",
    "mp-5986", "mp-504715", "mp-558125", "mp-19990", "mp-2998"]]
# RUN_LIST = "next_candidates.json"    # or a JSON file with the same entries

### The selected terms
What the choices above switch on: the calculations planned for every phase, the rules that apply, and the settings of each selected term.

In [ ]:
from heatup.core import contributions
contributions.show()

## Prepare
Copies the structures to `database/`, finds the competing phases of every chemical subsystem, and writes `plan.json` plus one folder per planned
calculation. Competing phases already in the database but computed with other settings are **refused** (`prepared.refused`). Safe to repeat.

In [ ]:
run_list = json.load(open(RUN_LIST)) if isinstance(RUN_LIST, str) else RUN_LIST
prepared = prepare_project(run_list)            # competitors=False: targets only
if prepared.missing:
    print("missing structures:", prepared.missing)
for tag, conflicts in prepared.refused.items():
    print("REFUSED", tag, {k: f"{a} -> {b}" for k, (a, b) in conflicts.items()})

## What will be computed
Supercells are taken from the *unrelaxed* structures (they can change by one repetition after relaxation): check them before spending GPU time.

In [ ]:
display(pd.DataFrame(preview_plan(prepared)).set_index("phase"))
show_status()  # everything is pending until notebook 02 (or your own code) has run

## Relax and write the cells
With `"relax"` in the gates every structure is relaxed now (finished ones are skipped); otherwise `CONTCAR` is a copy of `POSCAR`.
`export_inputs` then writes the cell of every planned calculation into its folder as a VASP `POSCAR` (phonon displacements, 12 strained cells,
MD supercell, quasi-harmonic volumes, ...). Notebook 02 builds exactly the same cells, so DFT and the potential see the same structures.

In [ ]:
if "relax" in cfg.plan.gates:
    relaxed = run_plan(steps=["relax"])
    display(pd.DataFrame(relaxed).groupby("action").size().rename("relaxations").to_frame())

exported = export_inputs(prepared)
display(pd.DataFrame(exported).set_index("phase"))

## Computing a step with another code
Run it on the cells in the folders and bring the results back; notebook 03 then treats them as `external`.

* `harmonic`: forces of every `phonons/harmonic/disp-*` cell -> `heatup.collect_harmonic(phase, forces, perfect_forces=...)`
* `elastic`: stresses of the cells in `elastic/strain/` (ions relaxed) -> `heatup.collect_elastic(phase, {"xx_plus": [...], ...})`
* `md`: `output.traj` (ASE, with momenta) and `md.json` in `md/<T>K/`
* structure and energy: `relaxation/CONTCAR` and `energy.json`

In [ ]:
save_project()          # heatup_project.json: folder roots + calculator, so notebooks 02 and 03 find this project
print_layout(cfg.phases.database_root, max_depth=3)